# Group 8 Demo: Code Defect Detection with LLMs and GraphCodeBERT

This notebook is a **class demo summary**. It does not retrain any model. Instead, it reads the completed experiment artifacts in `results/`, summarizes validation/test metrics, and visualizes the final comparison.

**Project question:** Are prompt-only LLMs enough for code defect detection, or do task adaptation and code-specific pretraining matter?

## 1. Demo Roadmap

Use this notebook for a 2-3 minute live walkthrough after the PPT:

1. Show the four evaluated approaches.
2. Load the final metrics from JSON files.
3. Compare Macro-F1 and Defective-F1.
4. Show confusion matrices.
5. State the final takeaway: supervised baselines are stronger than prompt-only LLMs; GraphCodeBERT has the best balanced test Macro-F1.

In [ ]:
from pathlib import Path
import json

import pandas as pd

try:
    from IPython.display import Image, display, Markdown
except ImportError:
    Image = None
    Markdown = None
    display = print
    print("IPython is not installed; rich display output will fall back to print().")

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib is not installed; bar chart cells will print tables only.")

ROOT = Path.cwd()
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"

print("Repo root:", ROOT)
print("Results folder exists:", RESULTS.exists())

## 2. Methods Compared

| Method | What it tests | Status |
|---|---|---|
| Qwen zero-shot | Raw LLM prompt-only reasoning | validation + test complete |
| Qwen 4-shot | In-context learning with examples | validation + test complete, but collapsed |
| Qwen LoRA fine-tuned | Supervised lightweight fine-tuning | validation + test complete |
| Majority baseline | Most frequent class sanity check | test complete |
| TF-IDF Linear SVM | Traditional lexical ML baseline | test complete |
| TF-IDF Logistic Regression | Traditional lexical ML baseline | test complete |
| GraphCodeBERT | Supervised code-specific encoder | validation + test complete |

In [ ]:
metric_files = {
    ("Qwen zero-shot", "validation"): RESULTS / "zero_shot_validation_metrics.json",
    ("Qwen zero-shot", "test"): RESULTS / "zero_shot_test_metrics.json",
    ("Qwen 4-shot", "validation"): RESULTS / "four_shot_validation_metrics.json",
    ("Qwen 4-shot", "test"): RESULTS / "four_shot_test_metrics.json",
    ("Qwen LoRA fine-tuned", "validation"): RESULTS / "lora_validation_metrics.json",
    ("Qwen LoRA fine-tuned", "test"): RESULTS / "lora_test_metrics.json",
    ("Majority baseline", "test"): RESULTS / "majority_test_metrics.json",
    ("TF-IDF Linear SVM", "test"): RESULTS / "tfidf_linear_svm_test_metrics.json",
    ("TF-IDF Logistic Regression", "test"): RESULTS / "tfidf_logreg_test_metrics.json",
    ("GraphCodeBERT", "validation"): RESULTS / "graphcodebert_validation_metrics.json",
    ("GraphCodeBERT", "test"): RESULTS / "graphcodebert_test_metrics.json",
}

rows = []
for (method, split), path in metric_files.items():
    with path.open("r", encoding="utf-8") as f:
        m = json.load(f)
    rows.append({
        "method": method,
        "split": split,
        "accuracy": m["accuracy"],
        "macro_f1": m["macro_f1"],
        "defective_f1": m["defective_f1"],
        "defective_recall": m["defective_recall"],
        "tn": m["tn"],
        "fp": m["fp"],
        "fn": m["fn"],
        "tp": m["tp"],
        "file": str(path.relative_to(ROOT)),
    })

df = pd.DataFrame(rows)
display(df.sort_values(["split", "macro_f1"], ascending=[True, False]))

## 3. Final Test-Set Comparison

For the final report/demo, emphasize the **test split** because it is the held-out performance comparison. Macro-F1 and Defective-F1 are more informative than accuracy for this task.

In [ ]:
test_df = df[df["split"] == "test"].copy()
test_df = test_df.sort_values("macro_f1", ascending=False)

display(test_df[["method", "accuracy", "macro_f1", "defective_f1", "defective_recall", "tn", "fp", "fn", "tp"]])

if plt is not None:
    ax = test_df.set_index("method")[["macro_f1", "defective_f1"]].plot(
        kind="bar",
        figsize=(9, 4.5),
        rot=20,
        color=["#2E74B5", "#70AD47"],
    )
    ax.set_title("Final Test Performance")
    ax.set_ylabel("F1 score")
    ax.set_ylim(0, 0.75)
    ax.grid(axis="y", alpha=0.25)
    for container in ax.containers:
        ax.bar_label(container, fmt="%.3f", fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print("Install matplotlib to render this bar chart, or use the table above for demo.")

## 4. Key Result Interpretation

- **GraphCodeBERT is the strongest balanced model**: test Macro-F1 `0.6517`, Defective-F1 `0.6017`.
- **TF-IDF Logistic Regression is surprisingly competitive**: test Macro-F1 `0.6218`, Defective-F1 `0.6088`.
- **LoRA fine-tuned Qwen is stable but weaker**: test Macro-F1 `0.5507`, Defective-F1 `0.5236`.
- **Zero-shot Qwen has limited signal**: test Macro-F1 `0.5180`.
- **4-shot Qwen collapsed**: Defective-F1 `0.0000` because it predicted all test samples as non-defective.

This supports the main conclusion: **for code defect detection, supervised models are much stronger than prompt-only examples, and code-aware modeling gives the best balanced result.**

## 5. Confusion Matrices

Rows are true labels and columns are predicted labels: `[[TN, FP], [FN, TP]]`.

The confusion matrices are useful for explaining the 4-shot/majority collapse and why supervised TF-IDF and GraphCodeBERT are more convincing.

In [ ]:
overview = FIGURES / "confusion_matrices_overview.png"
if overview.exists() and Image is not None:
    display(Image(filename=str(overview)))
elif overview.exists():
    print("Confusion matrix overview image:", overview)
else:
    print("Missing", overview)
    print("Run: python scripts/plot_confusion_matrices.py")

In [ ]:
for _, row in test_df.iterrows():
    matrix = [[row["tn"], row["fp"]], [row["fn"], row["tp"]]]
    print(f"{row['method']} test confusion matrix [[TN, FP], [FN, TP]]:")
    print(matrix)
    print()

## 6. Demo Summary

This section summarizes the main points for the live demo:

1. The project evaluates code defect detection as a binary classification task on CodeXGLUE.
2. The experiment compares a ladder of approaches: zero-shot prompting, few-shot prompting, LoRA fine-tuning, traditional TF-IDF baselines, and GraphCodeBERT.
3. Macro-F1 and Defective-F1 are the key metrics because accuracy alone can hide class collapse.
4. The 4-shot setting collapsed to non-defective predictions, showing that prompt-only examples were not reliable here.
5. GraphCodeBERT performed best on test Macro-F1, while TF-IDF Logistic Regression was highly competitive on Defective-F1.
6. The main takeaway is that supervised task adaptation and code-aware modeling are more effective than prompt-only LLM use for this task.

## 7. Reproducibility Pointers

- GitHub: https://github.com/JiufuZh/526
- Metrics: `results/*.json`
- CPU baseline test metrics: `results/majority_test_metrics.json`, `results/tfidf_linear_svm_test_metrics.json`, `results/tfidf_logreg_test_metrics.json`
- Final result table: `results/report_ready_metrics.md`
- Figures: `results/figures/`
- LLM evaluation script: `scripts/evaluate_llm.py`
- Encoder training script: `scripts/train_encoder_baseline.py`
- Encoder test evaluation script: `scripts/evaluate_encoder_baseline.py`
- GraphCodeBERT config: `configs/encoder_graphcodebert.yaml`
- Qwen LoRA config: `configs/lora_bf16_512.yaml`

In [ ]:
artifact_paths = [
    RESULTS / "report_ready_metrics.md",
    RESULTS / "graphcodebert_test_metrics.json",
    RESULTS / "lora_test_metrics.json",
    RESULTS / "zero_shot_test_metrics.json",
    RESULTS / "four_shot_test_metrics.json",
    FIGURES / "confusion_matrices_overview.png",
]

for path in artifact_paths:
    print(f"{path.relative_to(ROOT)}: {'OK' if path.exists() else 'MISSING'}")

## 8. References

- CodeXGLUE benchmark: https://github.com/microsoft/CodeXGLUE
- GraphCodeBERT paper: https://arxiv.org/abs/2009.08366
- GraphCodeBERT model card: https://huggingface.co/microsoft/graphcodebert-base
- Qwen model family: https://huggingface.co/Qwen
- LoRA paper: https://arxiv.org/abs/2106.09685
- Project code: https://github.com/JiufuZh/526